#1. Basic Tasks 

##1. Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table. 

In [0]:

CREATE OR REPLACE TABLE cyntexa_dev.bronze.customers_raw (
  customer_id    INT,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  segment        STRING,
  is_active      BOOLEAN
);

INSERT INTO cyntexa_dev.bronze.customers_raw
VALUES
  (1, 'John Smith',     'john.smith@email.com',     '555-0101', 'New York',    'NY', 'Retail',     true),
  (2, 'Jane Doe',       'jane.doe@email.com',       '555-0102', 'Los Angeles', 'CA', 'Wholesale',  true),
  (3, 'Bob Johnson',    'bob.johnson@email.com',    '555-0103', 'Chicago',     'IL', 'Retail',     true),
  (4, 'Alice Brown',    'alice.brown@email.com',    '555-0104', 'Houston',     'TX', 'Enterprise', true),
  (5, 'Charlie Wilson', 'charlie.wilson@email.com', '555-0105', 'Phoenix',     'AZ', 'Retail',     true);


CREATE OR REPLACE TABLE cyntexa_dev.silver.customers AS
SELECT
  customer_id,
  customer_name,
  email,
  phone,
  city,
  state,
  segment,
  is_active,
  current_timestamp() AS last_updated
FROM cyntexa_dev.bronze.customers_raw;

SELECT * FROM cyntexa_dev.silver.customers ORDER BY customer_id;


CREATE OR REPLACE TABLE cyntexa_dev.bronze.customers_staging (
  customer_id    INT,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  segment        STRING,
  is_active      BOOLEAN
);

INSERT INTO cyntexa_dev.bronze.customers_staging
VALUES
  (1, 'John Smith',     'john.smith@new-email.com',  '555-0101', 'Boston',  'MA', 'Enterprise', true),
  (3, 'Bob Johnson',    'bob.johnson@email.com',    '555-9999', 'Chicago', 'IL', 'Wholesale', true),   
  (5, 'Charlie Wilson', 'charlie.wilson@email.com', '555-0105', 'Phoenix', 'AZ', 'Retail',    false),  
  (6, 'Diana Prince',   'diana.prince@email.com',   '555-0106', 'Seattle', 'WA', 'Enterprise', true),
  (7, 'Edward Norton',  'edward.norton@email.com',  '555-0107', 'Denver',  'CO', 'Retail',     true);

MERGE WITH SCHEMA EVOLUTION INTO cyntexa_dev.silver.customers AS target
USING cyntexa_dev.bronze.customers_staging AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
  

SELECT * FROM cyntexa_dev.silver.customers ORDER BY customer_id;

##2. Grant SELECT on a table to one group and a masked/limited view to another group using Unity Catalog permissions. 



###Permission Management

#### Create User Groups:
```sql
-- Run these in Databricks Account Console or via API
-- Groups: data_engineers, analysts, executives, sales_us, sales_eu
```

#### Grant Permissions:
```sql
-- Schema-level permissions
GRANT USAGE ON SCHEMA dev.bronze TO `data_engineers`;
GRANT USAGE ON SCHEMA dev.silver TO `data_engineers`;
GRANT USAGE ON SCHEMA dev.gold TO `data_engineers`;
GRANT USAGE ON SCHEMA dev.gold TO `analysts`;  -- Read-only for analysts

-- Table-level permissions
GRANT SELECT ON TABLE dev.gold.customer_metrics TO `analysts`;
GRANT SELECT ON TABLE dev.gold.sales_summary TO `analysts`;
GRANT ALL PRIVILEGES ON TABLE dev.bronze.customers_raw TO `data_engineers`;

-- Secured view access
GRANT SELECT ON VIEW dev.gold.customers_masked TO `analysts`;
GRANT SELECT ON VIEW dev.gold.sales_secure TO `sales_us`;
GRANT SELECT ON VIEW dev.gold.sales_secure TO `sales_eu`;

-- Revoke direct access to sensitive tables
REVOKE SELECT ON TABLE dev.silver.customers_clean FROM `analysts`;
```

##3. Look up the DBU consumption for a compute resource you've been using and explain, in plain terms, what a DBU is billing for. 

In [0]:
SELECT 
  account_id,
  usage_date,
  sku_name,
  cloud,
  usage_unit,
  usage_quantity,
  usage_metadata.*
FROM system.billing.usage
WHERE usage_unit = 'DBU'
ORDER BY usage_date DESC, usage_quantity DESC

### What is a DBU (Databricks Unit)?

A **DBU** is Databricks' unit of processing capability. In plain terms:

- **What you're paying for**: The compute power (CPU, memory, and resources) that Databricks provisions to run your workloads
- **How it works**: Different workload types consume DBUs at different rates:
  - **Jobs Compute**: Running scheduled ETL jobs and batch processing
  - **All-Purpose Compute**: Interactive notebook development and ad-hoc queries  
  - **SQL Warehouses**: Running SQL queries and dashboards

Think of it like electricity: a DBU measures the "processing power" you use, similar to how kilowatt-hours measure electrical power consumption.

##4. Build a full SCD Type 2 table: implement the MERGE that closes out old records (setting end_date and is_current) and inserts new versions when a tracked column changes. 

In [0]:

CREATE OR REPLACE TABLE cyntexa_dev.silver.customers_scd2 (
  customer_id    INT,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  segment        STRING,
  is_active      BOOLEAN,
  start_date     TIMESTAMP,
  end_date       TIMESTAMP,
  is_current     BOOLEAN
);


INSERT INTO cyntexa_dev.silver.customers_scd2
SELECT 
  customer_id,
  customer_name,
  email,
  phone,
  city,
  state,
  segment,
  is_active,
  current_timestamp() AS start_date,
  NULL AS end_date,
  true AS is_current
FROM cyntexa_dev.bronze.customers_raw;

SELECT * FROM cyntexa_dev.silver.customers_scd2 ORDER BY customer_id, start_date;


CREATE OR REPLACE TABLE cyntexa_dev.bronze.customers_scd2_staging (
  customer_id    INT,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  segment        STRING,
  is_active      BOOLEAN
);

INSERT INTO cyntexa_dev.bronze.customers_scd2_staging
VALUES
  (1, 'John Smith',     'john.smith@updated.com',  '555-1111', 'Boston',    'MA', 'Enterprise', true),  
  (2, 'Jane Doe',       'jane.doe@email.com',      '555-0102', 'Los Angeles', 'CA', 'Wholesale',  true), 
  (3, 'Bob Johnson',    'bob.johnson@email.com',   '555-9999', 'Chicago',   'IL', 'Wholesale',  false), 
  (6, 'Diana Prince',   'diana.prince@email.com',  '555-0106', 'Seattle',   'WA', 'Enterprise', true);  

MERGE INTO cyntexa_dev.silver.customers_scd2 AS target
USING (
  SELECT 
    s.*,
    current_timestamp() AS merge_timestamp
  FROM cyntexa_dev.bronze.customers_scd2_staging s
) AS source
ON target.customer_id = source.customer_id 
   AND target.is_current = true

WHEN MATCHED AND (
  target.customer_name != source.customer_name OR
  target.email != source.email OR
  target.phone != source.phone OR
  target.city != source.city OR
  target.state != source.state OR
  target.segment != source.segment OR
  target.is_active != source.is_active
)
THEN UPDATE SET
  target.end_date = source.merge_timestamp,
  target.is_current = false

WHEN NOT MATCHED THEN INSERT (
  customer_id,
  customer_name,
  email,
  phone,
  city,
  state,
  segment,
  is_active,
  start_date,
  end_date,
  is_current
) VALUES (
  source.customer_id,
  source.customer_name,
  source.email,
  source.phone,
  source.city,
  source.state,
  source.segment,
  source.is_active,
  source.merge_timestamp,
  NULL,
  true
);


INSERT INTO cyntexa_dev.silver.customers_scd2
SELECT 
  s.customer_id,
  s.customer_name,
  s.email,
  s.phone,
  s.city,
  s.state,
  s.segment,
  s.is_active,
  current_timestamp() AS start_date,
  NULL AS end_date,
  true AS is_current
FROM cyntexa_dev.bronze.customers_scd2_staging s
INNER JOIN (
  SELECT customer_id
  FROM cyntexa_dev.silver.customers_scd2
  WHERE is_current = false
    AND end_date IS NOT NULL
    AND end_date >= date_sub(current_timestamp(), 1)  -- Changed in this run
  GROUP BY customer_id
) closed ON s.customer_id = closed.customer_id;

-- Step 6: View the results showing full history
SELECT 
  customer_id,
  customer_name,
  email,
  city,
  state,
  segment,
  start_date,
  end_date,
  is_current
FROM cyntexa_dev.silver.customers_scd2
ORDER BY customer_id, start_date;